# Solving the Telegraph Equation: Numerical Methods vs. Neural Networks

This notebook solves the **1-D telegraph equation** using two approaches:

1. **Numerical Methods** — Explicit Central Difference and Crank–Nicolson-style Implicit Scheme
2. **Neural Network** — Physics-Informed Neural Network (PINN) via PyTorch

---

## Problem Statement

The telegraph equation models signal propagation in lossy transmission lines, damped vibrations, and diffusion with finite propagation speed:

$$\frac{\partial^2 u}{\partial t^2} + 2\alpha\,\frac{\partial u}{\partial t} = c^2\,\frac{\partial^2 u}{\partial x^2}, \qquad x \in [0, L],\; t \in [0, T]$$

This is a **linear, second-order hyperbolic PDE** with constant wave speed $c$ and **damping coefficient** $\alpha > 0$. It interpolates between:
- the **wave equation** ($\alpha = 0$): pure propagation, no dissipation
- the **diffusion equation** (large $\alpha$, short time): parabolic-like behaviour

The damping term $2\alpha\,u_t$ causes exponential energy decay — signals attenuate as they propagate.

**Initial conditions** — first mode, released from rest:

$$u(x, 0) = \sin\!\left(\frac{\pi x}{L}\right), \qquad \frac{\partial u}{\partial t}(x, 0) = 0$$

**Boundary conditions** — fixed ends (Dirichlet):

$$u(0, t) = u(L, t) = 0$$

**Exact solution** — for $\alpha < \omega_0 = c\pi/L$ (under-damped regime), the first mode is a **decaying oscillation**:

$$u^*(x, t) = e^{-\alpha t}\sin\!\left(\frac{\pi x}{L}\right)\!\left[\cos(\omega_d\,t) + \frac{\alpha}{\omega_d}\sin(\omega_d\,t)\right]$$

where the **damped frequency** is:

$$\omega_d = \sqrt{\omega_0^2 - \alpha^2} = \sqrt{c^2\!\left(\frac{\pi}{L}\right)^{\!2} - \alpha^2}$$

The amplitude envelope decays as $e^{-\alpha t}$ while the string oscillates at the reduced frequency $\omega_d < \omega_0$. The total energy dissipates at rate $2\alpha$:

$$\frac{dE}{dt} = -2\alpha\int_0^L u_t^2\,dx \leq 0$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_banded
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
CMAP = "inferno"

# ---- Problem parameters ---------------------
C       = 1.0       # wave speed
ALPHA   = 0.5       # damping coefficient
L_DOM   = 1.0       # domain length [0, L]
T_END   = 3.0       # final time  (enough to see significant decay)

# Derived quantities
K1      = np.pi / L_DOM                             # first-mode wavenumber
OMEGA0  = C * K1                                    # undamped natural frequency
OMEGA_D = np.sqrt(OMEGA0**2 - ALPHA**2)             # damped frequency
T_PER   = 2 * np.pi / OMEGA_D                       # damped oscillation period

In [ ]:
def u_init(x):
    """Initial displacement: first mode sin(πx/L)."""
    return np.sin(K1 * x)


def v_init(x):
    """Initial velocity: released from rest."""
    return np.zeros_like(x)


def u_exact(x, t):
    """Exact solution: damped oscillating first mode."""
    return (np.exp(-ALPHA * t) * np.sin(K1 * x)
            * (np.cos(OMEGA_D * t) + (ALPHA / OMEGA_D) * np.sin(OMEGA_D * t)))


# Quick preview — exact solution at several times
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x_plot = np.linspace(0, L_DOM, 500)
for ax, t_snap in zip(axes, [0.0, 1.0, 2.0, 3.0]):
    ax.plot(x_plot, u_exact(x_plot, t_snap), "k-", lw=2)
    ax.fill_between(x_plot, u_exact(x_plot, t_snap), alpha=0.25, color="steelblue")
    ax.set_title(f"$t = {t_snap:.2f}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

plt.suptitle("Exact Solution — Telegraph Equation (Damped First Mode)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"c = {C},  α = {ALPHA},  L = {L_DOM}")
print(f"ω₀ = cπ/L = {OMEGA0:.4f},  ω_d = √(ω₀²−α²) = {OMEGA_D:.4f}")
print(f"Damped period = 2π/ω_d = {T_PER:.4f},  T_end = {T_END}")
print(f"Amplitude at t={T_END}: e^{{-αT}} = {np.exp(-ALPHA * T_END):.4f}  ({100*(1 - np.exp(-ALPHA * T_END)):.1f}% decay)")

---

## Part 1 — Numerical Methods

### 1-A. Explicit Central Difference

Using central differences in time for $u_{tt}$, forward difference for $u_t$ at level $n$, and central differences in space:

$$\frac{u_j^{n+1} - 2u_j^n + u_j^{n-1}}{\Delta t^2} + 2\alpha\,\frac{u_j^{n+1} - u_j^{n-1}}{2\,\Delta t} = c^2\,\frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{\Delta x^2}$$

Solving for $u_j^{n+1}$:

$$u_j^{n+1} = \frac{1}{1 + \alpha\Delta t}\left[2u_j^n - (1 - \alpha\Delta t)\,u_j^{n-1} + r^2\!\left(u_{j+1}^n - 2u_j^n + u_{j-1}^n\right)\right]$$

where $r = c\,\Delta t/\Delta x$.

**CFL stability condition**: $r \leq 1$ (same as for the undamped wave equation — damping does not destabilise the scheme).

**Bootstrap**: with zero initial velocity:

$$u_j^1 = u_j^0 + \frac{r^2}{2}\left(u_{j+1}^0 - 2u_j^0 + u_{j-1}^0\right) - \alpha\,\Delta t\,u_j^0 \cdot \frac{\Delta t}{2} \cdot 0$$

Since $u_t(x,0)=0$, the Taylor expansion gives:

$$u_j^1 = u_j^0 + \frac{\Delta t^2}{2}\left[c^2\,\frac{u_{j+1}^0 - 2u_j^0 + u_{j-1}^0}{\Delta x^2} - 2\alpha \cdot 0\right] = u_j^0 + \frac{r^2}{2}(u_{j+1}^0 - 2u_j^0 + u_{j-1}^0)$$

In [ ]:
def solve_explicit(Nx=200, Nt=600, T=T_END):
    """
    Explicit central-difference scheme for the telegraph equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)   Grid including boundary points.
    u_hist : list of ndarray   Solution snapshots.
    t_hist : list of float     Snapshot times.
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    r   = C * dt / dx

    assert r <= 1.0, f"CFL violated: r = {r:.4f} > 1"
    print(f"Explicit — dx={dx:.5f}, dt={dt:.6f}, Courant r={r:.4f}, αΔt={ALPHA * dt:.4e}")

    coeff_fwd = 1.0 / (1.0 + ALPHA * dt)
    coeff_old = 1.0 - ALPHA * dt

    # Level n=0
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    # Level n=1 via Taylor expansion (v_init = 0, so u_t(0)=0)
    u_curr       = u_prev.copy()
    u_curr[1:-1] = (u_prev[1:-1]
                    + 0.5 * r**2 * (u_prev[2:] - 2 * u_prev[1:-1] + u_prev[:-2]))
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # Explicit update with damping
        d2u          = u_curr[2:] - 2 * u_curr[1:-1] + u_curr[:-2]
        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = coeff_fwd * (2 * u_curr[1:-1]
                                    - coeff_old * u_prev[1:-1]
                                    + r**2 * d2u)
        u_next[0] = u_next[-1] = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_ex, Nt_ex                    = 200, 600
x_ex, u_ex_hist, t_ex_hist      = solve_explicit(Nx_ex, Nt_ex)

# Error at final time
u_ref_ex   = u_exact(x_ex, T_END)
err_ex     = np.abs(u_ex_hist[-1] - u_ref_ex)
print(f"Max absolute error  : {err_ex.max():.3e}")
print(f"Mean absolute error : {err_ex.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_ex_hist, t_ex_hist)):
    u_ref = u_exact(x_ex, t_snap)

    axes[0, col].plot(x_ex, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ex, u_snap, "b--", lw=1.5, label="Explicit")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_ex, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Explicit Central Difference — Telegraph Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1-B. Implicit (Crank–Nicolson-style) Scheme

We write the telegraph equation as $u_{tt} + 2\alpha\,u_t = \mathcal{L}u$ where $\mathcal{L}u = c^2 u_{xx}$. The implicit three-level scheme averages both the spatial operator *and* the damping term symmetrically:

$$\frac{u^{n+1} - 2u^n + u^{n-1}}{\Delta t^2} + 2\alpha\,\frac{u^{n+1} - u^{n-1}}{2\Delta t} = \frac{1}{4}\left(\mathcal{L}u^{n+1} + 2\,\mathcal{L}u^n + \mathcal{L}u^{n-1}\right)$$

Rearranging with $s = c^2\Delta t^2/(4\Delta x^2)$:

$$\left[(1 + \alpha\Delta t)\,I - s\,D^2\right]\mathbf{u}^{n+1} = 2\mathbf{u}^n - (1 - \alpha\Delta t)\,\mathbf{u}^{n-1} + s\,D^2\!\left(2\mathbf{u}^n + \mathbf{u}^{n-1}\right)$$

where $D^2$ is the second-difference matrix. The LHS is tridiagonal with constant coefficients:

| Band | LHS coefficient |
|------|----------------|
| sub/super | $-s$ |
| diagonal | $1 + \alpha\Delta t + 2s$ |

This scheme is **unconditionally stable** and **second-order accurate** in both space and time: $\mathcal{O}(\Delta t^2, \Delta x^2)$.

In [ ]:
def solve_implicit(Nx=200, Nt=200, T=T_END):
    """
    Implicit (Crank–Nicolson-style) scheme for the telegraph equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)
    u_hist : list of ndarray
    t_hist : list of float
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    N   = Nx            # number of interior points

    s   = C**2 * dt**2 / (4 * dx**2)
    print(f"Implicit — dx={dx:.5f}, dt={dt:.5f}, s={s:.4f}, αΔt={ALPHA * dt:.4f}")

    # LHS banded matrix: (1 + αΔt + 2s) on diag, -s on sub/super
    A_band          = np.zeros((3, N))
    A_band[0, 1:]   = -s                              # super-diagonal
    A_band[1, :]    = 1 + ALPHA * dt + 2 * s          # diagonal
    A_band[2, :-1]  = -s                              # sub-diagonal

    def D2(u_full):
        """Second spatial difference on interior points."""
        return (u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]) / dx**2

    def D2_raw(u_full):
        """Second spatial difference (without 1/dx²) on interior points."""
        return u_full[2:] - 2 * u_full[1:-1] + u_full[:-2]

    # Initialise: u^0 and bootstrap u^1
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    # Bootstrap: v_init=0, so u^1 ≈ u^0 + (Δt²/2)·c²·u_xx^0
    u_curr       = u_prev.copy()
    u_curr[1:-1] = u_prev[1:-1] + 0.5 * (C * dt)**2 * D2(u_prev)
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # RHS
        rhs = (2 * u_curr[1:-1]
               - (1 - ALPHA * dt) * u_prev[1:-1]
               + s * (2 * D2_raw(u_curr) + D2_raw(u_prev)))

        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = solve_banded((1, 1), A_band, rhs)
        u_next[0]    = 0.0
        u_next[-1]   = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


Nx_im, Nt_im = 200, 300
x_im, u_im_hist, t_im_hist = solve_implicit(Nx_im, Nt_im)

u_ref_im = u_exact(x_im, T_END)
err_im   = np.abs(u_im_hist[-1] - u_ref_im)
print(f"Max absolute error  : {err_im.max():.3e}")
print(f"Mean absolute error : {err_im.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_im_hist, t_im_hist)):
    u_ref = u_exact(x_im, t_snap)

    axes[0, col].plot(x_im, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_im, u_snap, "g--", lw=1.5, label="Implicit")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_im, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Implicit (CN-style) — Telegraph Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grid-refinement convergence study ----------------------------------------
Nx_list = [25, 50, 100, 200, 400]
err_ex_conv, err_im_conv = [], []

for Nx in Nx_list:
    # Explicit: ensure CFL ≤ 1
    dx_c = L_DOM / (Nx + 1)
    Nt_c = max(int(np.ceil(T_END * C / (0.9 * dx_c))), 300)
    x_c, u_c, t_c   = solve_explicit(Nx, Nt_c)
    u_ex_c           = u_exact(x_c, t_c[-1])
    err_ex_conv.append(np.max(np.abs(u_c[-1] - u_ex_c)))

    # Implicit — fixed moderate Nt
    x_c2, u_c2, t_c2 = solve_implicit(Nx, Nt=600)
    u_ex_c2           = u_exact(x_c2, t_c2[-1])
    err_im_conv.append(np.max(np.abs(u_c2[-1] - u_ex_c2)))

dh = [L_DOM / (Nx + 1) for Nx in Nx_list]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dh, err_ex_conv, "bs-", lw=1.5, label="Explicit")
ax.loglog(dh, err_im_conv, "ro-", lw=1.5, label="Implicit (CN-style)")
ax.loglog(dh, [h**2 * err_ex_conv[0] / dh[0]**2 for h in dh],
          "b--", lw=1.0, label="$\\mathcal{O}(h^2)$")
ax.loglog(dh, [h**2 * err_im_conv[0] / dh[0]**2 for h in dh],
          "r--", lw=1.0, label="$\\mathcal{O}(h^2)$ (ref)")
ax.set_xlabel("Grid spacing $h$")
ax.set_ylabel("Max absolute error at $t = T$")
ax.set_title("Convergence Study — FDM Schemes")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 2 — Neural Network: Physics-Informed Neural Network (PINN)

### How It Works

A PINN learns $\hat{u}_\theta(x, t)$ by minimising:

$$\mathcal{L} = \underbrace{\mathcal{L}_{IC}}_{\text{initial conditions}} + \underbrace{\mathcal{L}_{BC}}_{\text{boundary}} + \underbrace{\mathcal{L}_{PDE}}_{\text{physics residual}}$$

$$\mathcal{L}_{IC} = \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}(x_k, 0) - u_0(x_k)\Bigr)^2 + \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}_t(x_k, 0) - 0\Bigr)^2$$

$$\mathcal{L}_{BC} = \frac{1}{N_{BC}}\sum_k \left[\hat{u}(0, t_k)^2 + \hat{u}(L, t_k)^2\right]$$

$$\mathcal{L}_{PDE} = \frac{1}{N_f}\sum_k \left(\frac{\partial^2\hat{u}}{\partial t^2} + 2\alpha\,\frac{\partial\hat{u}}{\partial t} - c^2\,\frac{\partial^2\hat{u}}{\partial x^2}\right)^2$$

The PDE residual includes the **first-order damping term** $2\alpha\,u_t$ alongside the second derivatives $u_{tt}$ and $u_{xx}$. The first derivative $u_t$ is a by-product of computing $u_{tt}$ — no extra computational cost.

Like the wave and Klein–Gordon equations, the IC loss enforces *two* conditions — initial displacement **and** initial velocity — since the telegraph equation is second-order in time.

Training: **Adam** warm-up → **L-BFGS** fine-tuning.

In [ ]:
# -----------------------------------------------------------------
# Network Architecture
# -----------------------------------------------------------------
class TelegraphPINN(nn.Module):
    """Fully-connected PINN: (x, t) → u.
    Tanh activations ensure smooth second-order derivatives via autograd.
    """

    def __init__(self, hidden_layers=5, hidden_dim=80):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def grad1(u, v):
    return torch.autograd.grad(
        u, v, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]


# -----------------------------------------------------------------
# Collocation / IC / BC Points
# -----------------------------------------------------------------
N_IC    = 2000    # IC points (displacement + velocity)
N_BC    = 1000    # boundary points (each side)
N_INT   = 10000   # interior PDE collocation points

# ---- IC at t = 0 ----------------------------------
x_ic = (torch.rand(N_IC, 1) * L_DOM).requires_grad_(True)
t_ic = torch.zeros(N_IC, 1, requires_grad=True)
u_ic = torch.tensor(
    u_init(x_ic.detach().numpy()), dtype=torch.float32
)

# ---- Boundary points (x = 0 and x = L) -----------
t_bc = torch.rand(N_BC, 1) * T_END
x_bc_left  = torch.zeros(N_BC, 1)
x_bc_right = torch.full((N_BC, 1), L_DOM)

# ---- Interior PDE collocation points --------------
x_int = (torch.rand(N_INT, 1) * L_DOM).requires_grad_(True)
t_int = (torch.rand(N_INT, 1) * T_END).requires_grad_(True)

mse = nn.MSELoss()

print(f"IC  points : {N_IC}")
print(f"BC  points : {2 * N_BC}")
print(f"PDE points : {N_INT}")

In [ ]:
def compute_loss(model):
    # ---- IC loss: displacement + velocity ------
    u_pred_ic   = model(x_ic, t_ic)
    loss_ic_u   = mse(u_pred_ic, u_ic)

    u_t_ic      = grad1(u_pred_ic, t_ic)
    loss_ic_v   = mse(u_t_ic, torch.zeros_like(u_t_ic))

    loss_ic     = loss_ic_u + loss_ic_v

    # ---- Boundary loss (u = 0 at x = 0, L) -----
    u_left  = model(x_bc_left,  t_bc)
    u_right = model(x_bc_right, t_bc)
    loss_bc = mse(u_left,  torch.zeros_like(u_left)) + mse(u_right, torch.zeros_like(u_right))

    # ---- PDE residual: u_tt + 2α·u_t − c²·u_xx = 0 -----
    u_pred  = model(x_int, t_int)
    u_t     = grad1(u_pred, t_int)
    u_tt    = grad1(u_t,    t_int)
    u_x     = grad1(u_pred, x_int)
    u_xx    = grad1(u_x,    x_int)
    residual = u_tt + 2 * ALPHA * u_t - C**2 * u_xx
    loss_pde = mse(residual, torch.zeros_like(residual))

    return loss_ic + loss_bc + loss_pde, loss_ic, loss_bc, loss_pde


# ---- Phase 1: Adam --------------------------------
model = TelegraphPINN(hidden_layers=5, hidden_dim=80)
opt_adam = optim.Adam(model.parameters(), lr=1e-3)
ADAM_EP = 5000
history = []

for ep in range(1, ADAM_EP + 1):
    opt_adam.zero_grad()
    loss, l_ic, l_bc, l_pde = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if ep % 1000 == 0:
        print(f"[Adam] Ep {ep:5d} | Loss {loss.item():.5f} | "
              f"IC {l_ic.item():.5f} | BC {l_bc.item():.5f} | PDE {l_pde.item():.5f}")

print("Adam phase done.")

In [ ]:
# ---- Phase 2: L-BFGS ----
opt_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=500, tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn="strong_wolfe"
)


def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss(model)
    loss.backward()
    history.append(loss.item())
    return loss


opt_lbfgs.step(closure)
final, _, _, _ = compute_loss(model)
print(f"L-BFGS done.  Final loss: {final.item():.6f}")

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(history, color="steelblue", lw=1.0)
ax.axvline(x=ADAM_EP, color="tomato", ls="--", lw=1.5, label="Adam → L-BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss (log scale)")
ax.set_title("PINN Training Loss — Telegraph Equation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate PINN on a dense x grid at each snapshot time ----
model.eval()
Nev  = 500
x_ev = np.linspace(0, L_DOM, Nev)


def pinn_predict(t_val):
    xt = torch.tensor(x_ev, dtype=torch.float32).unsqueeze(1)
    tt = torch.full((Nev, 1), t_val, dtype=torch.float32)
    with torch.no_grad():
        return model(xt, tt).numpy().ravel()


snap_times_pinn = [0.0, T_END / 3, 2 * T_END / 3, T_END]
U_pinn_snaps    = [pinn_predict(t) for t in snap_times_pinn]

# PINN error at final time
U_ex_pinn   = u_exact(x_ev, T_END)
err_pinn    = np.abs(U_pinn_snaps[-1] - U_ex_pinn)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (U_snap, t_snap) in enumerate(zip(U_pinn_snaps, snap_times_pinn)):
    u_ref = u_exact(x_ev, t_snap)

    axes[0, col].plot(x_ev, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ev, U_snap, "m--", lw=1.5, label="PINN")
    axes[0, col].set_title(f"$t = {t_snap:.2f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(U_snap - u_ref)
    axes[1, col].plot(x_ev, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Physics-Informed Neural Network — Telegraph Equation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"PINN — Max error at t=T: {err_pinn.max():.2e}  |  Mean error: {err_pinn.mean():.2e}")

---

## Part 3 — Side-by-Side Comparison

Solutions from all three methods at $t = T$ alongside the exact solution, plus a quantitative error summary and amplitude decay check.

In [ ]:
# Interpolate FD solutions onto the common evaluation grid
from scipy.interpolate import interp1d

u_ex_ev     = interp1d(x_ex, u_ex_hist[-1], kind="linear",
                        bounds_error=False, fill_value=0.0)(x_ev)
u_im_ev     = interp1d(x_im, u_im_hist[-1], kind="linear",
                        bounds_error=False, fill_value=0.0)(x_ev)
u_pinn_ev   = U_pinn_snaps[-1]

err_ex_ev   = np.abs(u_ex_ev   - U_ex_pinn)
err_im_ev   = np.abs(u_im_ev   - U_ex_pinn)
err_pinn_ev = np.abs(u_pinn_ev - U_ex_pinn)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

labels  = ["Explicit", "Implicit (CN)", "PINN", "Exact"]
sols    = [u_ex_ev, u_im_ev, u_pinn_ev, U_ex_pinn]
colors  = ["steelblue", "seagreen", "mediumorchid", "black"]
errs    = [err_ex_ev, err_im_ev, err_pinn_ev]
e_lbl   = ["Explicit error", "Implicit error", "PINN error"]

for col, (lbl, sol, c) in enumerate(zip(labels, sols, colors)):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(x_ev, sol, color=c, lw=2)
    ax.fill_between(x_ev, sol, alpha=0.2, color=c)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

for col, (lbl, err) in enumerate(zip(e_lbl, errs)):
    ax = fig.add_subplot(gs[1, col])
    ax.plot(x_ev, err, "r-", lw=1.5)
    ax.set_title(f"{lbl}  (max {err.max():.1e})", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("|error|")
    ax.grid(alpha=0.3)

# Overlay plot for direct comparison
ax_ov = fig.add_subplot(gs[1, 3])
ax_ov.plot(x_ev, U_ex_pinn,  "k-",  lw=2,   label="Exact")
ax_ov.plot(x_ev, u_ex_ev,    "b--", lw=1.5, label="Explicit")
ax_ov.plot(x_ev, u_im_ev,    "g:",  lw=1.5, label="Implicit")
ax_ov.plot(x_ev, u_pinn_ev,  "m-.", lw=1.5, label="PINN")
ax_ov.set_title("$t = T$ overlay", fontsize=10)
ax_ov.set_xlabel("$x$")
ax_ov.set_ylabel("$u$")
ax_ov.legend(fontsize=8)
ax_ov.grid(alpha=0.3)

fig.suptitle(f"Final Comparison at $t = T = {T_END}$ — Telegraph Equation", fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- Quantitative error summary + amplitude decay check ----
def l2_norm(f_arr, x_arr):
      return np.sqrt(np.trapezoid(f_arr**2, x_arr))


# Peak amplitude at t=T (the max over x of the exact solution)
amp_exact = np.max(np.abs(U_ex_pinn))
amp_ex    = np.max(np.abs(u_ex_ev))
amp_im    = np.max(np.abs(u_im_ev))
amp_pinn  = np.max(np.abs(u_pinn_ev))

# Theoretical envelope at t=T
env_T = np.exp(-ALPHA * T_END)

print("=" * 72)
print(f"{'Method':<22} {'Max error':>12} {'Mean error':>12} {'L² error':>12} {'Peak |u|':>10}")
print("-" * 72)
print(f"{'Exact':22} {'—':>12} {'—':>12} {'—':>12} {amp_exact:>10.4f}")
print(f"{'Explicit':22} {err_ex_ev.max():>12.3e} {err_ex_ev.mean():>12.3e} "
      f"{l2_norm(err_ex_ev, x_ev):>12.3e} {amp_ex:>10.4f}")
print(f"{'Implicit (CN-style)':22} {err_im_ev.max():>12.3e} {err_im_ev.mean():>12.3e} "
      f"{l2_norm(err_im_ev, x_ev):>12.3e} {amp_im:>10.4f}")
print(f"{'PINN':22} {err_pinn_ev.max():>12.3e} {err_pinn_ev.mean():>12.3e} "
      f"{l2_norm(err_pinn_ev, x_ev):>12.3e} {amp_pinn:>10.4f}")
print("=" * 72)
print(f"\nDamping envelope: e^{{-αT}} = e^{{-{ALPHA}·{T_END}}} = {env_T:.4f}")
print(f"All methods should recover the decaying amplitude — peak |u(T)| ≈ {amp_exact:.4f}.")

---

## Summary

### About the Telegraph Equation

The telegraph equation is a fundamental PDE in electrical engineering, acoustics, and mathematical physics. Key properties:

- It is a **linear, second-order hyperbolic PDE** that adds a **first-order damping term** $2\alpha\,u_t$ to the wave equation. It models lossy transmission lines, damped vibrations, and Cattaneo heat conduction (finite-speed diffusion).
- Unlike the wave equation, the telegraph equation is **dissipative** — the total energy $E(t) = \frac{1}{2}\int[u_t^2 + c^2 u_x^2]\,dx$ decays monotonically at rate proportional to $\alpha$.
- In the **under-damped regime** ($\alpha < \omega_0 = c\pi/L$), solutions oscillate with exponentially decaying amplitude $\sim e^{-\alpha t}$. The damped frequency $\omega_d = \sqrt{\omega_0^2 - \alpha^2}$ is lower than the natural frequency.
- In the **over-damped regime** ($\alpha > \omega_0$), oscillations are replaced by pure exponential decay — the solution creeps back to equilibrium without oscillating.
- As $\alpha \to \infty$ (strong damping), the equation reduces to a **diffusion equation** $u_t \approx (c^2/2\alpha)\,u_{xx}$ — the transition from hyperbolic to parabolic behaviour.

### Method Comparison

| Aspect | Explicit (Central Diff.) | Implicit (CN-style) | PINN |
|--------|------------------------|---------------------|------|
| **Core idea** | Central differences + damping correction | Three-level implicit averaging | Minimise PDE + IC + BC residuals |
| **Accuracy** | $\mathcal{O}(\Delta t^2,\,h^2)$ | $\mathcal{O}(\Delta t^2,\,h^2)$ | Depends on training |
| **Stability** | Conditional (CFL: $r \leq 1$) | Unconditional | Unconditional |
| **Damping term** | Modifies update coefficients | Absorbed into tridiagonal LHS | Autograd — $u_t$ is free |
| **Energy decay** | Faithful (explicit dissipation) | Faithful (symmetric averaging) | Soft (through loss) |
| **Linear system** | None (explicit) | Tridiagonal solve per step | None (gradient descent) |
| **Mesh required** | Yes — uniform 1D grid | Yes — uniform 1D grid | No — meshfree |
| **Best for** | Quick prototyping | Large time steps, moderate damping | High-dimensional or inverse problems |

### Key Observations

- The **explicit scheme** handles the damping term via simple coefficient modifications: the coefficients $(1 + \alpha\Delta t)^{-1}$ and $(1 - \alpha\Delta t)$ scale the new and old time levels. The CFL condition $r \leq 1$ remains unchanged — damping does not affect the spatial stability constraint.
- The **implicit scheme** absorbs the damping into the tridiagonal LHS diagonal: $1 + \alpha\Delta t + 2s$. The added $\alpha\Delta t$ *improves* diagonal dominance, making the system better-conditioned compared to the undamped wave equation.
- The **PINN** naturally captures the damping term $2\alpha\,u_t$ since $u_t$ is already computed as an intermediate result when obtaining $u_{tt}$ via autograd. This makes the telegraph equation no harder than the wave equation for a PINN to handle.